In [ ]:
from typing_extensions import TypedDict, Literal, Annotated, Dict 
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.types import Command
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage
from pathlib import Path
import os
import json
import subprocess
from pathlib import Path

In [ ]:
WORKDIR = Path.cwd()


TOOLS = [
    {"name": "bash",       "description": "Run a shell command."},
    {"name": "read_file",  "description": "Read file contents."},
    {"name": "write_file", "description": "Write content to file."},
    {"name": "edit_file",  "description": "Replace text in file once."},
    {"name": "glob",       "description": "Find files by pattern."},
]

# Optimized system prompt to enforce cleaner JSON and explicit completion handling
SYSTEM = f"""You are a coding agent at {os.getcwd()}. Use the tools available to solve tasks.
You must interact with tools using valid JSON format. 

If you need to run a command, respond ONLY with a JSON array containing a tool_call block.
Example:
[
    {{"type": "tool_call", "tool": "bash", "input": {{"command": "echo 'hello'"}}}}
]

You are provided the following tools: {str(TOOLS)}

If the task has been completed and no more actions are needed, output exactly: STOP
Act, don't explain your reasoning. Do not wrap your JSON response in markdown code blocks."""

def safe_path(p: str) -> Path:
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path


def run_bash(command: str) -> str:
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(command, shell=True, cwd=WORKDIR,
                           capture_output=True, text=True, timeout=120)
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"


def run_read(path: str, limit: int = None) -> str:
    try:
        text = safe_path(path).read_text()
        lines = text.splitlines()
        if limit and limit < len(lines):
            lines = lines[:limit] + [f"... ({len(lines) - limit} more lines)"]
        return "\n".join(lines)[:50000]
    except Exception as e:
        return f"Error: {e}"


def run_write(path: str, content: str) -> str:
    try:
        fp = safe_path(path)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content)
        return f"Wrote {len(content)} bytes to {path}"
    except Exception as e:
        return f"Error: {e}"


def run_edit(path: str, old_text: str, new_text: str) -> str:
    try:
        fp = safe_path(path)
        content = fp.read_text()
        if old_text not in content:
            return f"Error: Text not found in {path}"
        fp.write_text(content.replace(old_text, new_text, 1))
        return f"Edited {path}"
    except Exception as e:
        return f"Error: {e}"
    
TOOL_HANDLERS = {
    "bash":       lambda **kw: run_bash(kw["command"]),
    "read_file":  lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file":  lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
}

def agent_call(messages: list[Dict[str, str]]):
    max_iter = 3 
    model = ChatOllama(model="gemma4", temperature=0.1)
    
    while max_iter > 0:
        history_str = ""
        for m in messages:
            history_str += f"\n{m['role'].capitalize()}: {m['content']}"

        prompt = f"System: {SYSTEM}\nTools: {str(TOOLS)}\nHistory: {history_str}"
        
        response = model.invoke(prompt)
        content = response.content if hasattr(response, "content") else response.get("content", "")
        content = content.strip()
        
        print(f"\n--- Iteration {4 - max_iter} ---")
        print("Agent Response Content:\n", content)
        
        if "stop" in content.lower():
            print("Stop signal received. Exiting loop.")
            messages.append({"role": "assistant", "content": "STOP"})
            break
        
        content_cleaned = content.replace("\xa0", " ")
        if content_cleaned.startswith("```"):
            content_cleaned = content_cleaned.split("\n", 1)[1] if "\n" in content_cleaned else content_cleaned
            content_cleaned = content_cleaned.rsplit("```", 1)[0].strip()
        else:
            content_cleaned = content_cleaned.strip()
            
        try: 
            blocks = json.loads(content_cleaned)
        except json.JSONDecodeError as e:
            print(f"JSON Parsing failed. Raw: {content_cleaned}")
            return f"Error: LLM did not return valid JSON. Error: {str(e)}. Output was: {content}"
        
        messages.append({"role": "assistant", "content": content_cleaned})
        
        tool_outputs = []
        for block in blocks:
            if block.get("type") == "tool_call":
                command = block.get("input", {}).get("command", "")
                print(f"Executing Bash: {command}")
                output = run_bash(command)
                tool_outputs.append(f"Tool ({block.get('tool', 'unknown')}) output: {output}")
        
        if tool_outputs:
            combined_output = "\n".join(tool_outputs)
            messages.append({"role": "tools", "content": combined_output})
        else:
            messages.append({"role": "tools", "content": "No tools were executed. Please continue or STOP."})
            
        max_iter -= 1
        
    return messages

# Run the fixed agent
history = [{"role": "user", "content": 'Create a file called hello.py that prints "Hello, World!"'}]
result = agent_call(history)